<a href="https://colab.research.google.com/github/chedy322/Vector-Search-Engine/blob/main/SemanticSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers
!pip install -U sentence-transformers

In [ ]:
# embend the data
from sentence_transformers import SentenceTransformer

# data
data=[
     "Cheap apartment in Riga city center",
    "Luxury villa with pool in Spain",
    "Affordable flat near Riga university",
    "Modern apartment in Berlin",
    "Budget room in Riga close to transport",
]

# Load the model
model=SentenceTransformer("all-MiniLM-L6-v2")
# encode the data
embeddings_data=model.encode(data)
# store the embendding in np
import numpy as np
np.save("embeddings.npy",embeddings_data)


In [ ]:
import numpy as np
# load the embedding
embeddings=np.load("embeddings.npy")
# user input
userInput="cheap apartments in Riga"
# convert user input into embedding
userInputEmbeddings=model.encode(userInput)
# simlairity search
from sklearn.metrics.pairwise import cosine_similarity
similaritySearchResult=cosine_similarity(
    [userInputEmbeddings],
    embeddings
)[0]

# return the statements
top_k=3
min_score=0.4

scores=np.argsort(similaritySearchResult)[::-1]
# print the statements
result=[]
for i in scores:
  if(similaritySearchResult[i]>=min_score):
    result.append((data[i],similaritySearchResult[i]))
result=result[:3]

for sen,score in result:
  print(f"{sen}----> {score:.4f}")





In [ ]:
# Step 2 Cross encode
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tested_results=[x[0] for x in result]
pairs=[(userInput,doc ) for doc in tested_results]
accuracy_score=reranker.predict(pairs)

convert_to_probabilty=lambda x: 1/(1+np.exp(-x))
accuracy_score=[convert_to_probabilty(x) for x in accuracy_score]

user_output=[]
for i in range(len(accuracy_score)):
  if(accuracy_score[i]>=min_score):
    # take the score fromt he result[i]
    result_score=result[i][1]
    result_sentence=result[i][0]
    # append the sentence equivalent to the probabilty to the user_output
    user_output.append(result_sentence)
